In [ ]:
import random
import numpy as np


def to_bits(x,n):
    return np.array([(x>>i) & 1 for i in range(n)],dtype=np.uint8)

def to_int(v):
    x=0
    for i,b in enumerate(v):
        x|=(int(b)&1)<<i
        return x
    
def rndm_bi_mat(n):
    return np.random.randint(0, 2, size=(n, n), dtype=np.uint8)


def gf2_rank(M):
    M=M.copy().astype(np.uint8)
    rs,cs = M.shape
    rk = 0
    col = 0
    for r in range(rs):
        pivot=None
        for i in range(r,rs):
            if M[i,col]==1:
                pivot = i
                break
        if pivot is None:
            col+=1
            if col >= cs:
                break
            r-=1
            continue
        if pivot != r:
            M[[r,pivot]] = M[[pivot,r]]
        for i in range(r+1,rs):
            if M[i,col]==1:
                M[i,:]^=M[r,:]
        rk +=1
        col +=1
        if col >= cs:
            break
    return rk

def rng_inv_mat(n):
    while True:
        M = rndm_bi_mat(n)
        if gf2_rank(M) == n:
            return M
        
def apply_lin(M,x,n):
    v=to_bits(x,n)
    y=M.dot(v)%2
    return to_int(y)

def rng_bit_perm(n):
    perm = list(range(n))
    random.shuffle(perm)
    return perm

def apply_perm(perm,x,n):
    v=to_bits(x,n)
    w = np.zeros_like(v)
    for new, old in enumerate(perm):
        w[new]=v[old]
    return to_int(w)

def gen_le_pair(n):
    N = 1<<n

    g1 = list(range(N))
    random.shuffle(g1)

    s= rng_inv_mat(n)

    p = rng_bit_perm(n)

    g2 = [0]*N
    for x in range(N):
        x_perm = apply_perm(p,x,n)
        y1=g1[x_perm]
        y2 = apply_lin(s,y1,n)
        g2[x]=y2 
    return (g1,g2,s,p)

if __name__ == "__main__":
    n=5
    g1,g2,s,p = gen_le_pair(n)
    print("g1:",g1)
    print("g2:",g2)
    print("s:\n",s)
    print("p:",p)

: 

In [119]:
import random
import numpy as np

# ---------- bit utilities ----------

def int_to_bits(x, n):
    """
    Map integer x in [0, 2^n) to an n-dimensional vector over GF(2),
    using little-endian bit order: v[0] = LSB.
    """
    return np.array([(x >> i) & 1 for i in range(n)], dtype=np.uint8)

def bits_to_int(v):
    """
    Inverse of int_to_bits under the same convention.
    """
    x = 0
    for i, b in enumerate(v):
        x |= (int(b) & 1) << i
    return x

# ---------- linear maps over GF(2) ----------

def random_binary_matrix(n):
    """
    Random n x n matrix over GF(2).
    """
    return np.random.randint(0, 2, size=(n, n), dtype=np.uint8)

def gf2_rank(M):
    """
    Compute rank of M over GF(2) via Gaussian elimination.
    """
    M = M.copy().astype(np.uint8)
    rows, cols = M.shape
    rank = 0
    col = 0
    for r in range(rows):
        if col >= cols:
            break

        # find a pivot row with 1 in column 'col'
        pivot = None
        for i in range(r, rows):
            if M[i, col] == 1:
                pivot = i
                break

        if pivot is None:
            # no pivot in this column; move to next column
            col += 1
            # stay at same r (outer loop will ++r, so undo here)
            r -= 1
            continue

        # swap pivot row into position r
        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]

        # eliminate below
        for i in range(r + 1, rows):
            if M[i, col] == 1:
                M[i, :] ^= M[r, :]

        rank += 1
        col += 1

    return rank

def random_invertible_matrix(n):
    """
    Sample uniformly from M_n(F_2) until we get a full-rank matrix,
    i.e. an element of GL(n,2).
    """
    while True:
        M = random_binary_matrix(n)
        if gf2_rank(M) == n:
            return M

def apply_linear(M, x, n):
    """
    Apply linear map represented by matrix M to the n-bit vector x.
    """
    v = int_to_bits(x, n)
    y = M.dot(v) % 2
    return bits_to_int(y)

# ---------- bit permutation ----------

def random_bit_perm(n):
    """
    Random permutation of bit positions {0,...,n-1}.
    """
    perm = list(range(n))
    random.shuffle(perm)
    return perm

def apply_bit_perm(perm, x, n):
    """
    Apply bit permutation 'perm' to n-bit integer x.
    perm[new_pos] = old_pos.
    """
    v = int_to_bits(x, n)
    w = np.zeros_like(v)
    for new_pos, old_pos in enumerate(perm):
        w[new_pos] = v[old_pos]
    return bits_to_int(w)

# ---------- generate linearly equivalent pair ----------

def generate_le_pair(n):
    """
    Generate:
      - g1: random permutation on {0,...,2^n-1}
      - g2: permutation linearly equivalent to g1 via S and P
      - S: random invertible n x n matrix over GF(2)
      - P: random bit permutation of {0,...,n-1}
    so that  g2[x] = S( g1( P(x) ) )  for all x.
    """
    N = 1 << n  # 2^n

    # 1) random base permutation g1
    g1 = list(range(N))
    random.shuffle(g1)

    # 2) random invertible output map S
    S = random_invertible_matrix(n)

    # 3) random input bit-permutation P
    P = random_bit_perm(n)

    # 4) define g2(x) = S( g1( P(x) ) )
    g2 = [0] * N
    for x in range(N):
        x_p = apply_bit_perm(P, x, n)    # P(x)
        y1  = g1[x_p]                    # g1(P(x))
        y2  = apply_linear(S, y1, n)     # S(g1(P(x)))
        g2[x] = y2

    # sanity checks
    assert len(set(g1)) == N, "g1 is not a permutation!"
    assert len(set(g2)) == N, "g2 is not a permutation!"
    assert gf2_rank(S) == n,  "S is not invertible!"
    assert sorted(P) == list(range(n)), "P is not a permutation of bit positions!"

    # verify the LE relation explicitly
    for x in range(N):
        lhs = g2[x]
        rhs = apply_linear(S, g1[apply_bit_perm(P, x, n)], n)
        assert lhs == rhs, "LE relation failed at x = {}".format(x)

    return g1, g2, S, P

if __name__ == "__main__":
    n = 5
    g1, g2, S, P = generate_le_pair(n)
    print("g1:", g1)
    print("g2:", g2)
    print("S:\n", S)
    print("P:", P)


g1: [3, 31, 11, 20, 30, 13, 23, 10, 9, 6, 5, 24, 21, 4, 8, 14, 25, 17, 15, 0, 29, 12, 19, 18, 16, 26, 27, 7, 28, 2, 22, 1]
g2: [12, 11, 1, 10, 5, 17, 22, 15, 28, 29, 8, 13, 25, 3, 0, 31, 30, 27, 18, 16, 21, 9, 6, 23, 24, 19, 26, 14, 7, 20, 4, 2]
S:
 [[0 0 1 1 0]
 [1 1 1 0 1]
 [0 1 0 0 0]
 [0 1 0 1 1]
 [0 0 1 0 0]]
P: [3, 2, 4, 0, 1]


In [ ]:
from collections import defaultdict
import itertools
import math

# -----------------------------
# Example codes
# -----------------------------
C  = [tuple(x) for x in [(1,1,0,0,0), (1,0,1,0,0)]]
Cp = [tuple(x) for x in [(1,0,1,0,0), (1,1,0,0,0)]]

# ============================================================
# 1) Incidence graph
# ============================================================
def build_incidence_graph(codewords):
    n = len(codewords[0])
    U = [("U", i) for i in range(len(codewords))]
    V = [("V", j) for j in range(n)]

    adj = defaultdict(set)

    for i, word in enumerate(codewords):
        for j, bit in enumerate(word):
            if bit == 1:
                adj[("U", i)].add(("V", j))
                adj[("V", j)].add(("U", i))

    return adj, U, V

# ============================================================
# 2) WL refinement
# ============================================================
def wl_trace(adj, U, V, max_iters=50):
    labels = {}

    for u in U:
        labels[u] = ("U", len(adj[u]))
    for v in V:
        labels[v] = ("V", len(adj[v]))

    for _ in range(max_iters):
        new_labels = {}
        for node in labels:
            neigh_labels = sorted(labels[nbr] for nbr in adj[node])
            new_labels[node] = (labels[node], tuple(neigh_labels))

        if new_labels == labels:
            break
        labels = new_labels

    return labels

# ============================================================
# 3) Align label IDs
# ============================================================
def aligned_ids(labels_C, labels_Cp):
    all_labels = set(labels_C.values()) | set(labels_Cp.values())
    label_to_id = {lab: i for i, lab in enumerate(sorted(all_labels))}
    return (
        {node: label_to_id[lab] for node, lab in labels_C.items()},
        {node: label_to_id[lab] for node, lab in labels_Cp.items()},
    )

# ============================================================
# 4) Extract coordinate classes
# ============================================================
def wl_coordinate_classes(V, idmap):
    classes = defaultdict(list)
    for v in V:
        classes[idmap[v]].append(v[1])
    return classes

# ============================================================
# 5) Apply permutation
# ============================================================
def apply_pi(codewords, pi):
    n = len(codewords[0])
    out = []
    for w in codewords:
        new_w = [0] * n
        for j in range(n):
            new_w[pi[j]] = w[j]
        out.append(tuple(new_w))
    return out

# ============================================================
# 6) WL-pruned permutation generator
# ============================================================
def wl_pruned_permutations_lazy(classes_src, classes_tgt, n, max_yield=None):
    blocks = []

    for k in classes_src:
        if len(classes_src[k]) != len(classes_tgt[k]):
            return
        blocks.append((classes_src[k], classes_tgt[k]))

    count = 0

    def backtrack(i, current_pi):
        nonlocal count
        if max_yield is not None and count >= max_yield:
            return
        if i == len(blocks):
            count += 1
            yield current_pi.copy()
            return

        src_block, tgt_block = blocks[i]
        for perm in itertools.permutations(tgt_block):
            new_pi = current_pi.copy()
            for s, t in zip(src_block, perm):
                new_pi[s] = t
            yield from backtrack(i + 1, new_pi)

    yield from backtrack(0, {})

# ============================================================
# 7) Driver
# ============================================================
def demo_verbose(C, Cp, MAX_LOG10=6.0, MAX_ENUM=200000):
    adjC, UC, VC = build_incidence_graph(C)
    adjCp, UCp, VCp = build_incidence_graph(Cp)

    labC = wl_trace(adjC, UC, VC)
    labCp = wl_trace(adjCp, UCp, VCp)

    idC, idCp = aligned_ids(labC, labCp)

    clsC = wl_coordinate_classes(VC, idC)
    clsCp = wl_coordinate_classes(VCp, idCp)

    # Estimate search size via logs
    log10_size = 0.0
    for k in clsC:
        m = len(clsC[k])
        log10_size += math.log10(math.factorial(m))

    if log10_size > MAX_LOG10:
        print("Search space too large (log10 estimate):", log10_size)
        return

    found = []
    for pi in wl_pruned_permutations_lazy(clsC, clsCp, len(VC), MAX_ENUM):
        mapped = apply_pi(C, pi)
        if sorted(mapped) == sorted(Cp):
            found.append(pi)

    print("Found permutations:", found)

# ============================================================
# Run
# ============================================================
demo_verbose(C, Cp)

# Orbits

In [4]:
import numpy as np
import scipy as si
import sympy as sm
from collections import defaultdict
# class DefineOrbitsProperties:
#     def __init__(self):
#         return self
    
def weight_frequency(self,C,S):
    C = np.array(C)
    return np.sum(np.all(C[:,S]==1, axis=1))
    
def orbits(C):
    C = np.array(C)
    n = C.shape[1]
    
    sigs = defaultdict(list)
    for j in range(n):
        col = tuple(C[:,j])
        sigs[col].append(j)
        
    return list(sigs.values())


        
     

In [4]:
import itertools
from collections import defaultdict
import numpy as np


def as_array(C):
    return np.array(list(C), dtype=int)


def coordinate_signature(C, j):
    C = as_array(C)
    n = C.shape[1]
    q_vals = sorted(set(C.flatten()))

    value_counts = tuple(int(np.sum(C[:, j] == a)) for a in q_vals)

    pairwise_per_k = []
    for k in range(n):
        counts = tuple(
            int(np.sum((C[:, j] == a) & (C[:, k] == b)))
            for a in q_vals for b in q_vals
        )
        pairwise_per_k.append(counts)

    # Sort so the signature doesn't depend on absolute column positions
    pairwise = tuple(sorted(pairwise_per_k))

    return (value_counts, pairwise)



def coordinate_blocks(C):
    """
    Partition coordinates by signature.
    Returns:
      blocks: dict signature -> list of coordinate indices
    """
    C = as_array(C)
    n = C.shape[1]
    blocks = defaultdict(list)

    for j in range(n):
        sig = coordinate_signature(C, j)
        blocks[sig].append(j)

    return dict(blocks)


def compatible_block_bijections(C1, C2):
    """
    Match blocks of C1 to blocks of C2 by identical signature.
    If signatures or block sizes disagree, no solution.
    """
    b1 = coordinate_blocks(C1)
    b2 = coordinate_blocks(C2)

    if set(b1.keys()) != set(b2.keys()):
        return None

    for sig in b1:
        if len(b1[sig]) != len(b2[sig]):
            return None

    return b1, b2


def permute_code(C, perm):
    
    return {tuple(row[j] for j in perm) for row in C}


def candidate_permutations_from_blocks(C1, C2):
    """
    Generate only permutations consistent with matching blocks.
    """
    matched = compatible_block_bijections(C1, C2)
    if matched is None:
        return

    b1, b2 = matched
    n = as_array(C1).shape[1]

    block_perms = []
    for sig in b1:
        src = b1[sig]
        tgt = b2[sig]
        local_maps = []
        for p in itertools.permutations(tgt):
            local_maps.append(dict(zip(src, p)))
        block_perms.append(local_maps)

    for choices in itertools.product(*block_perms):
        perm = list(range(n))
        for local in choices:
            for i, j in local.items():
                perm[i] = j
        yield tuple(perm)

def quick_invariants(C):
    C = as_array(C)
    weights = sorted(np.sum(C, axis=1))
    return weights


def find_equivalences(C1, C2):
    """
    Return all coordinate permutations found by block-pruned search
    such that pi(C1) = C2.
    """
    C2_set = {tuple(row) for row in as_array(C2)}
    solutions = []

    for perm in candidate_permutations_from_blocks(C1, C2):
        if permute_code(C1, perm) == C2_set:
            solutions.append(perm)

    return solutions
perm = (0, 2, 1, 4,3)
C1 = {
    (0,0,0,0,0),
    (1,1,0,0,1)
}

C2 = {
    (0,0,0,0,0),
    (1,0,1,1,0)
}
print(permute_code(C1, (0, 2, 1, 4,3)))
print(permute_code(C1, (0, 2, 1, 4,3)) == C2)
quick_invariants(C1) != quick_invariants(C2)
sols = find_equivalences(C1, C2)
print(sols)
print()

# All permutations taking C1 → C2
equiv = find_equivalences(C1, C2)
print("C1 → C2:", equiv)
print()

# All permutations taking C1 → C1 (automorphism group)
auts = find_equivalences(C1, C1)
print("Aut(C1):", auts)
print()

# Same for C2
auts2 = find_equivalences(C2, C2)
print("Aut(C2):", auts2)
print()

n = 5
all_perms = list(itertools.permutations(range(n)))
print("All n! perms:", len(all_perms))  # 6 for n=3

print()
brute = [p for p in all_perms if permute_code(C1, p) == {tuple(r) for r in as_array(C2)}]
print("Brute force C1→C2:", brute)
print()

# Should match the block-pruned result
# print("brute:", sorted(brute))
print("equiv:", sorted(equiv))
print()
print("in brute not equiv:", set(brute) - set(equiv))
print()
print("in equiv not brute:", set(equiv) - set(brute))
print()


{(1, 0, 1, 1, 0), (0, 0, 0, 0, 0)}
True
[(0, 2, 1, 4, 3), (0, 2, 4, 1, 3), (0, 3, 1, 4, 2), (0, 3, 4, 1, 2)]

C1 → C2: [(0, 2, 1, 4, 3), (0, 2, 4, 1, 3), (0, 3, 1, 4, 2), (0, 3, 4, 1, 2)]

Aut(C1): [(0, 1, 2, 3, 4), (0, 1, 3, 2, 4), (0, 4, 2, 3, 1), (0, 4, 3, 2, 1), (1, 0, 2, 3, 4), (1, 0, 3, 2, 4), (1, 4, 2, 3, 0), (1, 4, 3, 2, 0), (4, 0, 2, 3, 1), (4, 0, 3, 2, 1), (4, 1, 2, 3, 0), (4, 1, 3, 2, 0)]

Aut(C2): [(0, 1, 2, 3, 4), (0, 4, 2, 3, 1), (0, 1, 3, 2, 4), (0, 4, 3, 2, 1), (2, 1, 0, 3, 4), (2, 4, 0, 3, 1), (2, 1, 3, 0, 4), (2, 4, 3, 0, 1), (3, 1, 0, 2, 4), (3, 4, 0, 2, 1), (3, 1, 2, 0, 4), (3, 4, 2, 0, 1)]

All n! perms: 120

Brute force C1→C2: [(0, 2, 1, 4, 3), (0, 2, 4, 1, 3), (0, 3, 1, 4, 2), (0, 3, 4, 1, 2), (1, 2, 0, 4, 3), (1, 2, 4, 0, 3), (1, 3, 0, 4, 2), (1, 3, 4, 0, 2), (4, 2, 0, 1, 3), (4, 2, 1, 0, 3), (4, 3, 0, 1, 2), (4, 3, 1, 0, 2)]

equiv: [(0, 2, 1, 4, 3), (0, 2, 4, 1, 3), (0, 3, 1, 4, 2), (0, 3, 4, 1, 2)]

in brute not equiv: {(4, 2, 1, 0, 3), (4, 3, 1, 0, 2), (1, 2

In [1]:
import itertools
import numpy as np
from collections import defaultdict


def fq_star(q):
    """Nonzero elements of F_q (prime q)."""
    return list(range(1, q))

def diagonal_group(q, n):
    """
    (F_q^*)^n : all n-tuples of nonzero field elements.
    Size: (q-1)^n
    """
    return list(itertools.product(fq_star(q), repeat=n))

def symmetric_group(n):
    """S_n : all permutations of {0,...,n-1}. Size: n!"""
    return list(itertools.permutations(range(n)))

def monomial_group(q, n):
    """
    Mon_n(F_q) = (F_q^*)^n x S_n
    Each element is (d, pi) where:
      d  in (F_q^*)^n  — coordinate scalings
      pi in S_n        — coordinate permutation
    Size: (q-1)^n * n!
    Note: for q=2, F_2^* = {1}, so Mon_n(F_2) = S_n (no nontrivial scaling).
    """
    return list(itertools.product(diagonal_group(q, n), symmetric_group(n)))

# -------------------------------------------------------
# Action of a monomial on a vector / code
# -------------------------------------------------------

def apply_monomial(v, d, pi, q):
    """
    Apply monomial (d, pi) to vector v over F_q (prime q).
    result[i] = d[i] * v[pi[i]]  (mod q)
    """
    return tuple((d[i] * v[pi[i]]) % q for i in range(len(v)))

def apply_monomial_to_code(C, d, pi, q):
    """Apply monomial (d, pi) to every codeword in C (set of tuples)."""
    return frozenset(apply_monomial(v, d, pi, q) for v in C)

# -------------------------------------------------------
# Orbit of a generator matrix G in Mat_{k x n}(F_q)
# Rows of G are treated as codewords.
# -------------------------------------------------------

def orbit_of_code(G, q):
    """
    G   : numpy array of shape (k, n) over F_q
    q   : prime field size
    Returns the orbit {M · G : M in Mon_n(F_q)} as a set of frozensets.
    Each element of the orbit is a frozenset of codeword tuples.
    """
    k, n = G.shape
    C = frozenset(tuple(row) for row in G)
    M = monomial_group(q, n)

    orbit = set()
    for d, pi in M:
        orbit.add(apply_monomial_to_code(C, d, pi, q))
    return orbit

# -------------------------------------------------------
# Quick demo
# -------------------------------------------------------
q, n = 5,5
print(f"F_{q}^* = {fq_star(q)}")
print(f"|(F_{q}^*)^{n}| = {len(diagonal_group(q, n))}  (should be {(q-1)**n})")
print(f"|S_{n}|         = {len(symmetric_group(n))}  (should be {__import__('math').factorial(n)})")
print(f"|Mon_{n}(F_{q})| = {len(monomial_group(q, n))}  (should be {(q-1)**n * __import__('math').factorial(n)})")
print()

# Example: a 2x3 code over F_3
G = np.array([
    [1, 0, 1],
    [0, 1, 2],
    [1,0,2]
], dtype=int)
orb = orbit_of_code(G, q)
print(f"Orbit size of G under Mon_{n}(F_{q}): {len(orb)}")


F_5^* = [1, 2, 3, 4]
|(F_5^*)^5| = 1024  (should be 1024)
|S_5|         = 120  (should be 120)
|Mon_5(F_5)| = 122880  (should be 122880)

Orbit size of G under Mon_5(F_5): 384
